In [7]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from utils.utils import ComputeROIs

Operating system:  Linux


Autosaving every 180 seconds
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
##########################################################
##########################################################
##########################################################
def phase_correlation(a, b):
    G_a = np.fft.fft2(a)
    G_b = np.fft.fft2(b)
    conj_b = np.ma.conjugate(G_b)
    R = G_a*conj_b
    R /= np.absolute(R)
    r = np.fft.ifft2(R).real
    return r

In [9]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
#fname = '/home/cat/data/donato/bscope_tests/8499/data/Image_001_001.raw'
fname = '/home/cat/data/donato/bscope_tests/8499/image_1000frames.raw'
#fname = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

data = np.fromfile(fname, dtype='uint16').reshape(-1, 512,512)
print (data.shape)

plt.figure()
plt.imshow(data[10])
plt.show()


(1000, 512, 512)


In [52]:
#################################################################################################
############ FOLLOW SUITE2P: BUILD TEMPLATE FROM HIGHEST CORRELATED FRAMES FIRST ################
#################################################################################################
from tqdm import trange, tqdm

# find best correlation map first
idx_imgs = np.arange(200,600,1)

# make temporary template to match to
template = np.mean(data[idx_imgs],axis=0)

#
corr_maxs = np.zeros(idx_imgs.shape[0])
ctr=0
for k in tqdm(idx_imgs):

    #
    temp = phase_correlation(data[k], template)

    r,c = np.unravel_index(temp.argmax(), temp.shape)

    maxcorr = temp[r,c]
    #corr_maxs.append(maxcorr)
    corr_maxs[ctr] = maxcorr
    #print ("location of r,c: ", r,c, ", peak value: ", temp[r,c])
    ctr+=1
#
idx = np.argsort(corr_maxs)[::-1]

#
n_best_imgs = 100
idx_best = idx[:n_best_imgs]
template = data[idx_imgs[idx_best]].mean(0)

# create buffer around template
template[:50] = 0
template[-50:] = 0
template[:,:50] = 0
template[:,-50:] = 0


#####
plt.figure()
ax=plt.subplot(1,1,1)
plt.plot(corr_maxs)
plt.plot(corr_maxs[idx])
plt.title("Phase correlation between single frame and "+str(idx_imgs.shape[0]) +"average (curve depicts ordered correlations)")


#####
plt.figure()
ax=plt.subplot(1,2,1)
temp = data[idx_imgs].mean(0)
plt.title("Average map over "+str(idx_imgs.shape[0])+" images")
plt.imshow(temp,
          vmin=0,
          vmax=1500)


#
n_best_imgs = 100

ax=plt.subplot(1,2,2)
plt.title("Average map over highest correlated "+str(n_best_imgs)+" images")
plt.imshow(template,
          vmin=0,
          vmax=1500)

#
plt.show()


100%|████████████████████████████████████████████████████████████████████| 400/400 [00:17<00:00, 22.27it/s]


In [55]:
########################################################################
############### NOW CAN RUN THE PHASE CORRELATION CODE ##################
########################################################################
def pad_data(img):
    img[:50] = 0
    img[-50:] = 0
    img[:,:50] = 0
    img[:,-50:] = 0

    return img

#
drifts = []
maxcorrs = []
for k in trange(data.shape[0]):
    
    temp = pad_data(data[k])
    temp = phase_correlation(temp, template)

    r,c = np.unravel_index(temp.argmax(), temp.shape)

    maxcorr = temp[r,c]
    
    #
    maxcorrs.append(maxcorr)
    
    #
    drifts.append([r,c])
print ("DONE")


100%|██████████████████████████████████████████████████████████████████| 1000/1000 [00:45<00:00, 22.07it/s]

DONE


In [62]:
# test the phase correlation function
img1 = template.copy()
img2 = template.copy()

#
img2 = np.roll(im

#
temp = phase_correlation(img1, img2)

#
r,c = np.unravel_index(temp.argmax(), temp.shape)

maxcorr = temp[r,c]

#
print ("max corr: ", maxcorr)
print ("shifts: ", r,c)


max corr:  1.0
shifts:  0 0


In [61]:
##
plt.figure()
ax=plt.subplot(2,1,1)
plt.plot(maxcorrs)
plt.ylim(bottom=0)
plt.title("Max correlations...")

ax=plt.subplot(2,1,2)
temp = np.vstack(drifts)
plt.scatter(np.arange(temp.shape[0]),
            temp[:,0], label = 'x shifts')
plt.scatter(np.arange(temp.shape[0]),
            temp[:,1], label='y shifts')
plt.legend()
plt.show()

In [20]:
#####################################################
#####################################################
#####################################################


#
from scipy import misc
from matplotlib import pyplot
import numpy as np

#Get two images with snippet at different locations
im1 = np.mean(misc.face(), axis=-1) #naive colour flattening  

im2 = np.zeros_like(im1)    
im2[:200,:200] = im1[200:400, 500:700]

corrimg = phase_correlation(im1, im2)
r,c = np.unravel_index(corrimg.argmax(), corrimg.shape)
print ("location of r,c: ", r,c, ", peak value: ", corrimg[r,c])

#
plt.figure()
ax=plt.subplot(1,3,1)
plt.imshow(im1)
plt.scatter(c,r,c='red')


ax2=plt.subplot(1,3,2)
plt.imshow(im2)


#pyplot.figure(figsize=[8,8])
ax3=plt.subplot(1,3,3)
plt.imshow(corrimg, cmap='gray')

plt.show()

location of r,c:  200 500 , peak value:  0.21144702395054743
